<a href="https://colab.research.google.com/github/varshayinithangaraju30-lgtm/customer-segmentation-kmeans/blob/main/unit_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score

URL = "https://raw.githubusercontent.com/varshayinithangaraju30-lgtm/customer-segmentation-kmeans/main/Mall_Customers.csv"
FEATURES = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]

# ---------- Functions to test ----------
def load_and_scale(path):
    df = pd.read_csv(path).drop_duplicates().dropna(subset=FEATURES)
    return df, StandardScaler().fit_transform(df[FEATURES])

def evaluate_clustering(X, labels):
    if len(set(labels)) < 2:
        raise ValueError("Need at least 2 clusters to evaluate")
    return {"silhouette": silhouette_score(X, labels),
            "calinski_harabasz": calinski_harabasz_score(X, labels)}

def reduce_to_2d(X):
    if X.shape[1] < 2:
        raise ValueError("Need at least 2 features for PCA")
    pca = PCA(n_components=2, random_state=42)
    return pca.fit_transform(X), pca

def plot_clusters(X_2d, labels):
    fig, ax = plt.subplots()
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels)
    ax.set_xlabel("Principal Component 1")
    ax.set_ylabel("Principal Component 2")
    ax.set_title("Customer Segments (PCA 2D view)")
    return fig

# ---------- Test data ----------
df, X = load_and_scale(URL)
labels = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X)

# ---------- Tests: evaluation metrics ----------
def test_metrics_keys():
    scores = evaluate_clustering(X, labels)
    assert "silhouette" in scores and "calinski_harabasz" in scores

def test_silhouette_range():
    s = evaluate_clustering(X, labels)["silhouette"]
    assert -1 <= s <= 1

def test_calinski_positive():
    assert evaluate_clustering(X, labels)["calinski_harabasz"] > 0

def test_single_cluster_error():
    try:
        evaluate_clustering(X, np.zeros(len(X), dtype=int))
        assert False, "Should have raised error"
    except ValueError:
        pass

# ---------- Tests: visualization ----------
def test_pca_shape():
    X_2d, _ = reduce_to_2d(X)
    assert X_2d.shape == (X.shape[0], 2)

def test_pca_variance():
    _, pca = reduce_to_2d(X)
    assert 0 < pca.explained_variance_ratio_.sum() <= 1.0001

def test_pca_one_feature_error():
    try:
        reduce_to_2d(X[:, :1])
        assert False, "Should have raised error"
    except ValueError:
        pass

def test_plot_returns_figure():
    X_2d, _ = reduce_to_2d(X)
    fig = plot_clusters(X_2d, labels)
    assert isinstance(fig, matplotlib.figure.Figure)
    assert fig.axes[0].get_xlabel() == "Principal Component 1"
    plt.close(fig)

# ---------- Run all tests ----------
tests = [test_metrics_keys, test_silhouette_range, test_calinski_positive,
         test_single_cluster_error, test_pca_shape, test_pca_variance,
         test_pca_one_feature_error, test_plot_returns_figure]

passed = 0
for t in tests:
    try:
        t()
        print("PASS:", t.__name__)
        passed += 1
    except Exception as e:
        print("FAIL:", t.__name__, "->", e)

print(f"\n{passed}/{len(tests)} tests passed")

PASS: test_metrics_keys
PASS: test_silhouette_range
PASS: test_calinski_positive
PASS: test_single_cluster_error
PASS: test_pca_shape
PASS: test_pca_variance
PASS: test_pca_one_feature_error
PASS: test_plot_returns_figure

8/8 tests passed
